# Week 17, Day 4: MAP estimation: priors as regularisers

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w017/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Ridge equals MAP

With noise sd σ = 0.5 and prior sd τ = 0.3, compute `lam`, the ridge solution `w_ridge` in closed form, and `w_map` by running gradient descent on the negative log-posterior (5,000 steps of size one over the largest curvature) from zero. Check they agree.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(1996)
X = rng.normal(size=(40, 6)); w_true = rng.normal(0, 0.3, 6)
y = X @ w_true + 0.5 * rng.normal(size=40)
sigma, tau = 0.5, 0.3

lam = ...
w_ridge = ...

def neg_log_post_grad(w):
    return ...                                     # gradient of |y - Xw|^2 / (2 sigma^2) + |w|^2 / (2 tau^2)

w_map = np.zeros(6)
step = 1 / (np.linalg.eigvalsh(X.T @ X).max() / sigma ** 2 + 1 / tau ** 2)   # safe step size, 1 / largest curvature
for _ in range(5000):
    w_map = w_map - step * neg_log_post_grad(w_map)
print("ridge:", w_ridge.round(4)); print("MAP:  ", w_map.round(4)); print("truth:", w_true.round(4))

In [ ]:
import numpy as np
def test_same():
    "the ridge solution and the minimiser of the negative log-posterior agree to within 1e-6"
    assert np.abs(w_ridge - w_map).max() < 1e-6

def test_lambda():
    "lam equals sigma^2 / tau^2"
    assert abs(lam - 0.5 ** 2 / 0.3 ** 2) < 1e-12

test_same()
test_lambda()
print("All checks passed")

### Exercise: Three estimates of a receptive field

A 30-pixel receptive field is a smooth Gabor-like profile. With 50 white-noise trials and noise sd 1, estimate it by least squares, by ridge (λ = 10) and with the smoothness prior (λ = 30, D the first-difference matrix). Store the relative squared errors |ŵ − w|² / |w|² in the dictionary `err` with keys 'ols', 'ridge' and 'smooth'.

In [ ]:
rng = np.random.default_rng(3)                 # this exercise's own random numbers
P, n = 30, 50
pix = np.arange(P)
w_rf = np.exp(-(pix - 15) ** 2 / (2 * 4 ** 2)) * np.cos(2 * np.pi * (pix - 15) / 10)
Xs = rng.normal(size=(n, P))
ys = Xs @ w_rf + rng.normal(size=n)
D = np.diff(np.eye(P), axis=0)                 # (P - 1) x P first differences

est = {
    "ols": np.linalg.lstsq(Xs, ys, rcond=None)[0],
    "ridge": ...,
    "smooth": ...,
}
err = {k: np.sum((v - w_rf) ** 2) / np.sum(w_rf ** 2) for k, v in est.items()}
print({k: round(v, 3) for k, v in err.items()})
for k, v in est.items(): plt.plot(v, label=k)
plt.plot(w_rf, "k", lw=3, label="true"); plt.legend(); plt.xlabel("pixel"); plt.show()

In [ ]:
import numpy as np
def test_order():
    "the smoothness prior beats ridge, which beats least squares, in error against the true receptive field"
    assert err["smooth"] < err["ridge"] < err["ols"]

def test_big_gain():
    "the smoothness prior cuts the least-squares error by more than half"
    assert err["smooth"] < 0.5 * err["ols"]

test_order()
test_big_gain()
print("All checks passed")